# 1. Import Dataset

We are using a dataset from Hugging Face for phishing emails: https://huggingface.co/datasets/zefang-liu/phishing-email-dataset/viewer

First, we are importing the dataset directly:

In [ ]:
! pip install -q datasets huggingface_hub transformers torch scikit-learn

In [ ]:
from datasets import load_dataset

dataset = load_dataset("zefang-liu/phishing-email-dataset")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv:   0%|          | 0.00/52.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

# 2. Preprocessing
Preprocessing the dataset -- this is a shared standardized processing choice.

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset['train'])
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df = df[["Email Text", "Email Type"]]
df.columns = ["text", "label"]
df.head()

,text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,the other side of * galicismos * * galicismo *...,Safe Email
2,re : equistar deal tickets are you still avail...,Safe Email
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df["label"] = df["label"].apply(lambda x: 1 if x == "Phishing Email" else 0)
df.head()

,text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",0
1,the other side of * galicismos * * galicismo *...,0
2,re : equistar deal tickets are you still avail...,0
3,\nHello I am your hot lil horny toy.\n I am...,1
4,software at incredibly low prices ( 86 % lower...,1


The code below standarizes text so the model does not memorize safe IDs, accounts, urls, etc.

In [ ]:
import pandas as pd
import re
from collections import defaultdict

def clean_email_text(text):
    """Basic text cleaning for duplicate detection."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()      # normalize spaces
    return text

def further_clean(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"\S+@\S+", " <EMAIL> ", text)
    text = re.sub(r"\b\d+\b", " <NUM> ", text)
    text = re.sub(r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b", " <DATE> ", text)
    text = re.sub(r"\b(account|invoice|order|id|number)\s*\w*", r"\1 <ID>", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
df["text"] = df["text"].apply(clean_email_text)
df["text"] = df["text"].apply(further_clean)

df.head()


,text,label
0,re <NUM> <NUM> disc uniformitarianism re <NUM>...,0
1,the other side of galicismos galicismo is a sp...,0
2,re equistar deal tickets are you still availab...,0
3,hello i am your hot lil horny toy i am the one...,1
4,software at incredibly low prices <NUM> lower ...,1


In [ ]:
df = df.drop_duplicates()
df = df.dropna()
print(len(df))

17136


In [ ]:
df["text_length"] = df["text"].str.split().str.len()

df.groupby("label")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,10691.0,547.966327,24214.441023,0.0,76.5,163.0,324.5,2502955.0
1,6445.0,263.149263,495.822992,0.0,60.0,117.0,256.0,11751.0


In [ ]:
df = df[df["text_length"] <= 500]
df = df[df["text_length"] > 20]

df.groupby("label")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,8741.0,172.383137,119.145083,21.0,75.0,143.0,245.00,500.0
1,5306.0,142.676027,110.229270,21.0,62.0,107.0,186.75,500.0


In [ ]:
len(df)

14047

# 3. Splitting Dataset

In [ ]:
from sklearn.model_selection import train_test_split

x = df["text"]
y = df["label"]

# split into train(80) and temp (20)
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y,
    random_state=3,
    test_size=0.2,
    stratify=y)

print(x_train.shape)
print(x_temp.shape)

(11237,)
(2810,)


In [ ]:
x_valid, x_test, y_valid, y_test = train_test_split(
    x_temp, y_temp,
    random_state=3,
    test_size=0.5,
    stratify=y_temp)

print(x_valid.shape)
print(x_test.shape)

(1405,)
(1405,)


# 4. Load Tokenizer and Model

In [ ]:
#average amount of tokens -> important for max_length parameter in tokenizer

texts = df["text"].to_list()
avg_word_num = []

for t in texts[:3]:
    words = t.split()
    avg_word_num.append(len(words))

print(sum(avg_word_num)/len(avg_word_num))
print(max(avg_word_num))

154.33333333333334
210


We concluded that the maximum length we could use for models like BERT is 305.

In [ ]:
def set_seed(seed_value=42):
    import random
    import numpy as np
    import torch

    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

In [ ]:
# bert-base-uncased
from datasets import Dataset
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

# 5. Tokenize
We're tokenizing each train, validation, and test sets.

In [ ]:
x_train_list = x_train.to_list()
x_valid_list = x_valid.to_list()
x_test_list = x_test.to_list()

y_train_list = y_train.to_list()
y_valid_list = y_valid.to_list()
y_test_list = y_test.to_list()

y_train_tensor = torch.tensor(y_train.to_list())
y_valid_tensor = torch.tensor(y_valid.to_list())
y_test_tensor = torch.tensor(y_test.to_list())

train_encodings = tokenizer(x_train_list, truncation=True, padding=True, max_length=305, return_tensors='pt')
valid_encodings = tokenizer(x_valid_list, truncation=True, padding=True, max_length=305, return_tensors='pt')
test_encodings = tokenizer(x_test_list, truncation=True, padding=True, max_length=305, return_tensors='pt')


train_dataset = MyDataset(train_encodings, y_train_tensor)
valid_dataset = MyDataset(valid_encodings, y_valid_tensor)
test_dataset = MyDataset(test_encodings, y_test_tensor)


print(train_dataset[:5])

{'input_ids': tensor([[  101,  2047,  2338,  ...,  8909,  1028,   102],
        [  101,  2006, 10722,  ...,     0,     0,     0],
        [  101,  1042,  2860,  ...,     0,     0,     0],
        [  101,  2128,  2622,  ...,  1026, 16371,   102],
        [  101,  2182,  2003,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([0, 0, 0, 0, 1])}


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results_bert',
    # 3 passes through the entire dataset
    num_train_epochs=3,
    # 8 values in each batch for training data
    per_device_train_batch_size=8,
    # 8 values in each batch for evaluation data
    per_device_eval_batch_size=8,
    # keep learning rate small to better find minimum error
    learning_rate = 2e-5,
    # record updates every 100 steps
    logging_dir='./logs_bert',
    logging_steps = 100,
    eval_strategy = "steps",
    eval_steps = 100,
    report_to = "none"
)
#set trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset = valid_dataset
)
#train the model
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss,Validation Loss
100,0.402937,0.156214
200,0.192851,0.159208
300,0.161566,0.103693
400,0.075882,0.082654
500,0.144229,0.060260
600,0.108400,0.068741
700,0.073399,0.070645
800,0.135636,0.105630
900,0.085694,0.209797
1000,0.070223,0.064571


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4215, training_loss=0.051306188757627186, metrics={'train_runtime': 3345.3339, 'train_samples_per_second': 10.077, 'train_steps_per_second': 1.26, 'total_flos': 5283729922086900.0, 'train_loss': 0.051306188757627186, 'epoch': 3.0})

In [ ]:
# Save the model after training
model_path = "./fine_tuned_bert_phishing"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print("Model saved to: ", {model_path})

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to:  {'./fine_tuned_bert_phishing'}


In [ ]:
# Evaluate on validation data

import numpy as np
from sklearn.metrics import classification_report, accuracy_score

predictions = trainer.predict(valid_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print(f"Accuracy score: {accuracy_score(true_labels, preds)}")

print(classification_report(true_labels,
                            preds,
                            target_names = ["Safe Email (0)", "Phishing Email (1)"]))


Accuracy score: 0.9921708185053381
                    precision    recall  f1-score   support

    Safe Email (0)       1.00      0.99      0.99       874
Phishing Email (1)       0.98      1.00      0.99       531

          accuracy                           0.99      1405
         macro avg       0.99      0.99      0.99      1405
      weighted avg       0.99      0.99      0.99      1405



In [ ]:
# evaluate on test data